In [1]:
import os, re
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/"
BASE += "Project3_Procurement/"
F = BASE + "kenya_contract_awards.csv"
raw = pd.read_csv(F)
df = raw.copy()
AMT = "Supplier Contract Amount (USD)"
tot0 = raw[AMT].sum()
print(raw.shape, round(tot0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(4839, 21) 2145458626


In [2]:
print(df.isna().sum()[lambda s: s > 0])
print(df.duplicated().sum())
k = ["WB Contract Number", "Supplier ID"]
print(df.duplicated(k).sum())
print(df[AMT].quantile(
    [0.5, 0.9, 0.99, 0.999, 1]).round(0))
print((df[AMT] == 0).sum(), (df[AMT] < 0).sum())
print(df.nlargest(5, AMT)[
    ["Supplier", "Procurement Category",
     "Procurement Method", AMT]])
for c in df.select_dtypes("object"):
    s = df[c].dropna()
    a = (s != s.str.strip()).sum()
    b = s.str.contains(r"\s{2,}").sum()
    if a or b:
        print(c, a, b)
print(df["Contract Signing Date"].head(3))

Project Global Practice            163
Contract Description                 5
Supplier                             1
Supplier Country / Economy          21
Supplier Country / Economy Code     21
Review type                          3
dtype: int64
0
0
0.500        27847.0
0.900       535114.0
0.990      5468671.0
0.999     58815152.0
1.000    172294315.0
Name: Supplier Contract Amount (USD), dtype: float64
1 0
                                               Supplier Procurement Category  \
6     SINOHYDRO CORPORATION LIMITED (中国水利水电建设集团国际工程有...                Works   
1376         China State Construction Finance Co., Ltd.                Works   
4219            UNITED NATIONS CHILDREN'S FUND (UNICEF)                Goods   
1378         China State Construction Finance Co., Ltd.                Works   
3431           LONGJIAN ROAD AND BRIDGE COMPANY LIMITED                Works   

     Procurement Method  Supplier Contract Amount (USD)  
6      Request for Bids                    1.722

In [3]:
g = df.groupby("Supplier")["Supplier ID"].nunique()
m = g[g > 1].sort_values(ascending=False)
print(df["Supplier"].nunique(),
      df["Supplier ID"].nunique(), len(m))
print(m.head(5))
ics = df["Procurement Method"].eq(
    "Individual Consultant Selection")
print(ics.sum(), df.loc[ics, "Supplier"].nunique(),
      df.loc[ics, "Supplier ID"].nunique())

3017 3734 47
Supplier
INDIVIDUAL CONSULTANT               669
KRYPTONITE INTERNATIONAL LIMITED      5
VILAKAZI LIMITED                      3
ARAMOT ENTERPRISES                    2
AESA EAST AFRICA LTD                  2
Name: Supplier ID, dtype: int64
688 1 669


In [4]:
# Names, tiers and dates; the date format is now explicit (month/day/year)
df["supplier"] = df["Supplier"].fillna("UNKNOWN")
df["is_individual"] = ics
SUF = {"LTD", "LIMITED", "LLC", "INC", "PLC",
       "CO", "COMPANY", "CORP", "CORPORATION"}
def norm(s):
    s = str(s).upper().replace("&", " AND ")
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    w = s.split()
    while w and w[-1] in SUF:
        w.pop()
    return " ".join(w)
df["name_key"] = df["supplier"].map(norm)
M = {
 "Direct Selection": "Direct",
 "UN Agencies (Direct)": "Direct",
 "Request for Bids": "Full",
 "Quality And Cost-Based Selection": "Full",
 "Quality Based Selection": "Full",
 "Fixed Budget Selection": "Full",
 "Least Cost Selection": "Full",
 "Request for Quotations": "Limited",
 "Consultant Qualification Selection": "Limited",
 "Individual Consultant Selection": "Limited"}
df["method"] = (df["Procurement Method"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip())
df["tier"] = df["method"].map(M)
assert df["tier"].notna().all(), "tier map"
d = pd.to_datetime(df["Contract Signing Date"],
                   format="%m/%d/%Y", errors="coerce")
print(d.isna().sum(), d.min(), d.max())
fy = d.dt.year + (d.dt.month >= 7)
print((fy == df["Fiscal Year"]).mean())
df["country"] = df["Supplier Country / Economy Code"]
df["is_kenyan"] = df["country"].eq("KE")
df["zero_amt"] = df[AMT] == 0

0 2019-07-03 00:00:00 2026-10-15 00:00:00
1.0


In [5]:
v = (df[~df["is_individual"]]
     .groupby("name_key")[AMT].sum()
     .sort_values(ascending=False))
print(v.head(30).round(0))
w = pd.Series(v.index)
f2 = w.str.split().str[:2].str.join(" ")
c = pd.DataFrame({"first2": f2, "key": w})
c = c[f2.duplicated(keep=False)]
print(c.sort_values("first2").head(60))

name_key
CHINA STATE CONSTRUCTION FINANCE                                251365793.0
SINOHYDRO                                                       196302752.0
UNITED NATIONS CHILDREN S FUND UNICEF                           144158703.0
LONGJIAN ROAD AND BRIDGE                                         66048524.0
CHINA STATE CONSTRUCTION ENGINEERING                             57416816.0
JIANGXI CONSTRUCTION ENGINEERING GROUP                           46446011.0
SHANDONG HIGHWAY BRIDGE CONSTRUCTION GROUP BRIDGE TECHNOLOGY     43315459.0
HALANE CONSTRUCTION                                              37243538.0
ZHONGMEI ENGINEERING GROUP                                       32772363.0
CHAJU BUILDERS                                                   26195430.0
CHINA INTERNATIONAL WATER AND ELECTRIC                           24978234.0
SINOTEC GTSCC JOINT VENTURE                                      18552157.0
SOLITON TELMEC                                                   17133261.0
THE

In [6]:
# Hand-written merge map: clear variants of one company only (no JVs, dealers or related firms)
MERGE = {
    "ADCC INTERNATIONAL EA": "ADCC INTERNATIONAL EAST AFRICA LIMTED",
    "CFAO MOBILITY KENTA": "CFAO MOBILITY",
    "CFAO MOBILITY KENYA": "CFAO MOBILITY",
    "CFAO MOTORS": "CFAO MOTORS KENYA",
    "CFAO MOTORS K": "CFAO MOTORS KENYA",
    "ALPEX CONSULTING AFRICA LIMITED ACAL": "ALPEX CONSULTING AFRICA",
    "ALPEX CONSULTING AFRICA LTD ACAL": "ALPEX CONSULTING AFRICA",
    "AMPC INTERNATIONAL HEALTH CONSULTANTS AMPC": "AMPC INTERNATIONAL HEALTH CONSULTANT",
    "GLOGRA GENERAL SUPPLIERS CONTRACTORS": "GLOGRA GENERAL SUPPLIERS AND CONTRACTORS",
    "GO SOLAR SYSTEM": "GO SOLAR SYSTEMS",
    "HELLO TRACTOR K": "HELLO TRACTOR",
    "CHINA INTERNATIONAL WATER ELECTRIC": "CHINA INTERNATIONAL WATER AND ELECTRIC",
    "CHINA INTERNATIONAL WATER AND ELECTRIC CORP CWE": "CHINA INTERNATIONAL WATER AND ELECTRIC",
    "TOYOTA K": "TOYOTA KENYA"}
print("keys not found:", [k for k in MERGE if k not in set(df["name_key"])])
empty = df["name_key"] == ""
print("empty keys:", empty.sum())
df.loc[empty, "name_key"] = "ID " + df.loc[empty, "Supplier ID"].astype(str)
before = df["name_key"].copy()
df["name_key"] = df["name_key"].replace(MERGE)
print(len(MERGE), (before != df["name_key"]).sum())
df["supplier_key"] = np.where(
    df["is_individual"], "INDIVIDUAL CONSULTANT",
    df["name_key"] + " | " + df["country"].fillna("XX"))

keys not found: []
empty keys: 0
14 22


In [7]:
t = df[AMT].sum()
for c in ["Procurement Category", "tier",
          "Fiscal Year", "Review type"]:
    g = df.groupby(c, dropna=False)[AMT].agg(
        ["size", "sum"])
    g["share"] = g["sum"] / t
    print(g.sort_values("sum", ascending=False)
          .round(3))
print(df["is_individual"].mean().round(3),
      (df.loc[df["is_individual"], AMT].sum()
       / t).round(3))
s = (df[~df["is_individual"]]
     .groupby("supplier_key")[AMT].sum()
     .sort_values(ascending=False))
print(len(s), (s.head(10).sum() / s.sum()).round(3))
print(s.head(5).round(0))

                         size           sum  share
Procurement Category                              
Works                     777  1.327031e+09  0.619
Goods                    2576  4.471697e+08  0.208
Consultant Services      1301  3.460069e+08  0.161
Non-consulting Services   185  2.525060e+07  0.012
         size           sum  share
tier                              
Full     1078  1.712193e+09  0.798
Direct    387  2.913005e+08  0.136
Limited  3374  1.419652e+08  0.066
             size           sum  share
Fiscal Year                           
2022          877  5.925423e+08  0.276
2020          730  4.172481e+08  0.194
2024          499  3.926544e+08  0.183
2025          497  2.979070e+08  0.139
2021         1179  1.926774e+08  0.090
2026          299  1.195688e+08  0.056
2023          680  9.583786e+07  0.045
2027           78  3.702264e+07  0.017
             size           sum  share
Review type                           
Prior         379  1.507526e+09  0.703
Post        

In [8]:
clean = pd.DataFrame({
 "contract_no": df["WB Contract Number"],
 "supplier_key": df["supplier_key"],
 "country": df["country"],
 "is_kenyan": df["is_kenyan"].astype(int),
 "is_individual": df["is_individual"].astype(int),
 "category": df["Procurement Category"],
 "method": df["method"],
 "tier": df["tier"],
 "amount_usd": df[AMT],
 "zero_amt": df["zero_amt"].astype(int),
 "fiscal_year": df["Fiscal Year"],
 "signing_date": d.dt.strftime("%Y-%m-%d"),
 "review_type": df["Review type"].fillna("Unknown")})
assert len(clean) == 4839, "rows"
assert abs(clean["amount_usd"].sum() - tot0) < 1
assert clean["category"].notna().all()
assert clean["review_type"].eq("Prior").sum() == 379
assert clean["review_type"].eq("Post").sum() == 4457
clean.to_parquet(BASE + "awards_clean.parquet",
                 index=False)
clean.to_csv(BASE + "awards_clean.csv", index=False)
print(os.listdir(BASE))
print(round(os.path.getsize(
    BASE + "awards_clean.csv") / 1e6, 2), "MB")

['kenya_contract_awards.csv', 'dash', 'awards_clean.parquet', 'awards_clean.csv', 'queries.sql']
0.55 MB


Cleaning log: Kenya contract awards (p3_01_cleaning)

1. Raw kept untouched; worked on a copy. 4,839 rows, 21 columns, total value USD 2,145,458,626.
2. Duplicates: 0 duplicate rows and 0 duplicate contract-and-supplier pairs. 4,711 distinct contract numbers,
   so a contract with several suppliers spans several rows (one row = one supplier's share).
3. Missing values: Project Global Practice 163, Contract Description 5, Supplier 1, Supplier Country 21,
   Review type 3. Decision: kept. The missing supplier became "UNKNOWN"; the 21 rows with no country
   (USD 11.7M) are reported separately; the 3 missing review types became "Unknown".
4. Amounts: median 27,847, 99th percentile 5.47M, 99.9th percentile 58.8M, maximum 172.3M. 1 zero amount
   (the row with no supplier, flagged zero_amt) and 0 negative amounts.
5. Five largest contracts, all looking real: Sinohydro 172.3M, China State Construction Finance 146.5M and
   104.9M (all Works, Request for Bids), UNICEF 116.3M (Goods, Direct Selection) and Longjian Road and Bridge
   66.0M (Works, Request for Bids). Together 28.2% of total value. Decision: kept; a few large contracts
   dominate, and this is stated in the limits.
6. Whitespace: double spaces in Procurement Method (129 rows), Project Name (64), Supplier (4) and Contract
   Description (283), and leading or trailing spaces in Contract Description (440) and Borrower Contract
   Reference Number (5). Procurement Method fixed (spaces collapsed and trimmed); the other columns are not
   used in the analysis.
7. Method spelling: "Consultant Qualification  Selection" (double space, 129 rows) merged with
   "Consultant Qualification Selection" (2 rows), leaving 10 method labels.
8. Competition tiers: direct (Direct Selection, UN Agencies (Direct)); full (Request for Bids, Quality And
   Cost-Based Selection, Quality Based Selection, Fixed Budget Selection, Least Cost Selection); limited or
   simplified (Request for Quotations, Consultant Qualification Selection, Individual Consultant Selection).
   All 4,839 rows mapped. The tiers are my reading of the method names.
9. Dates: Contract Signing Date read as month/day/year with an explicit format, 0 unreadable, 3 Jul 2019 to
   15 Oct 2026. Fiscal year derived from the date (July start) matches the file's Fiscal Year in 100% of
   rows. FY2027 is partial.
10. Individual consultants: 688 rows (14.2% of rows, 1.3% of value) share one placeholder name across 669
    supplier IDs. Set aside under the label INDIVIDUAL CONSULTANT, so no personal names appear anywhere.
11. Supplier names: 3,017 names against 3,734 IDs, 47 names with more than one ID. Automatic normalisation
    (uppercase, punctuation removed, legal suffixes such as LTD and LIMITED stripped) turned 3,018 names
    (including UNKNOWN) into 2,941 keys. 0 keys were empty after normalising.
12. Merge map written by hand: 14 entries, 22 rows changed (Toyota, CFAO, Alpex, AMPC, ADCC, Glogra,
    Go Solar, Hello Tractor and China International Water and Electric spellings). Joint ventures, Egerton
    University units and dealers were not merged. Supplier key = cleaned name + supplier country; individuals
    use their own label. Final firm keys: 2,932, or 2,931 without the zero-value UNKNOWN row.
13. UNICEF (UN agency, Direct Selection, USD 144.2M, 6.7% of value) stays in the totals and the competition
    shares. Concentration is also reported without it, because buying through a UN agency is a different
    channel from buying from a market supplier. Country code 1W covers three UN agencies (UNICEF, UN-Habitat,
    UNOPS; USD 150.2M, 7.0% of value); all of them are direct selection.
14. Related companies and countries: China State Construction Finance (251.4M) and China State Construction
    Engineering (57.4M) stay separate keys; together they hold 14.4% of total value. One variant of China
    International Water and Electric is registered in Kenya (1 row, USD 0.58M) and stays a separate key.
    With the final map, 180 of 2,931 firms hold 80% of firm spend and the top 10 hold 42.6%.
15. Reconciled: 4,839 rows, total amount unchanged, 379 Prior + 4,457 Post + 3 Unknown review types.
    Saved awards_clean.parquet and awards_clean.csv in Project3_Procurement.